# Lesson 02 - Изследване на рамката Microsoft Agent

**Рамката Microsoft Agent (MAF)** е унифицирана рамка за създаване на AI агенти. Тя предоставя чиста, композираема архитектура с четири основни градивни блока:

- **Client** – свързва се с крайна точка на AI модел и управлява комуникацията
- **Agent** – обвива клиент с инструкции и дефиниции на инструменти
- **Tools** – разширяват възможностите на агента с персонализирани функции, които моделът може да извиква
- **Session** – поддържа история на разговора за многостранни взаимодействия

В този урок ще изградим **агент за резервация на пътувания**, който проверява наличността на дестинации, използвайки тези концепции.


## Настройка


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Разбиране на архитектурата на Agent Framework

Microsoft Agent Framework следва слоеста архитектура:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Клиент** – `AzureAIProjectAgentProvider` се свързва с внедряване на Azure OpenAI. Той обработва удостоверяване, форматиране на заявки и парсване на отговори.
2. **Agent** – Създаден от клиента чрез `provider.create_agent()`, агентът комбинира достъп до модела с инструкции (системен подканващ текст) и инструменти.
3. **Инструменти** – Python функции, декорирани с `@tool`, които агентът може да извиква, за да изпълнява действия или да извлича данни.
4. **Сесия** – Обект `AgentSession` (създаден чрез `agent.create_session()`), който съхранява историята на разговора, което позволява многоходова беседа, при която агентът помни предишния контекст.

Нека изградим всеки слой стъпка по стъпка.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Добавяне на инструменти с декоратора @tool

Инструментите позволяват на агентите да предприемат действия извън генерирането на текст. Декораторът `@tool` превръща обикновена Python функция в нещо, което агентът може да извика.

Основни точки:
- Използвайте `Annotated[type, "description"]`, за да разбере моделът всеки параметър.
- Документацията се превръща в описание на инструмента, което вижда моделът.
- `approval_mode="never_require"` означава, че инструментът се изпълнява автоматично без потвърждение от потребителя.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Създаване на агент с инструменти

Сега комбинираме клиента, инструкциите и инструментите в агент. `Instructions` действат като системно подсещане — те определят личността и поведението на агента.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Многоходови разговори със сесии

`AgentSession` (създадена чрез `agent.create_session()`) следи всички съобщения в разговор. Като подаваме същата сесия на всяко извикване на `agent.run()`, агентът има достъп до цялата история на разговора и може да се позовава на предишни съобщения.

Предаваме `tools=[check_destination_availability]`, за да може агентът да използва нашия проверител на наличност по време на всеки ход.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Обобщение

В този урок изследвахте четирите стълба на Microsoft Agent Framework:

| Концепция | Какво научихте |
|-----------|----------------|
| **Клиент** | `AzureAIProjectAgentProvider` се свързва с Azure OpenAI с удостоверяване на база креденшъли |
| **Агент** | `provider.create_agent()` свързва модел с инструкции и име |
| **Инструменти** | Декораторът `@tool` предоставя Python функции за извикване от агента |
| **Сесия** | `agent.create_session()` поддържа история на разговора през няколко стъпки |

Тези градивни елементи се комбинират, за да създадат агенти, които могат да водят естествени разговори, да извикват външни функции и да поддържат контекст — основата за по-усъвършенствани агентски модели в по-късните уроци.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:  
Този документ е преведен с помощта на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, имайте предвид, че автоматичните преводи може да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Не носим отговорност за никакви недоразумения или погрешни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
